# Easy Italian News
- download and clean up episodes
- download mp3 files as well
- this is good for 1 month (can change date to do it for other months)

# Find all days with news

In [1]:
import requests
from bs4 import BeautifulSoup
import re

l_to_remove = [#r'\n+', 
    r'www\..*\n',
    r'Click the link..*\n', 
    r'https://..*\n', 
    r'\xa0\n', 
    r'Creative Commons..*\n',
    r'Image.*\n', 
    # r'it\.[\w+]..*\n', 'tg24\..*\n',
    # r'[\s]Jeffrey Zeldman\n',
    # r'[\s]Zdravko Petrov\n', 
    # r'[\s]Chris Watt\n',
    # r'[\s]GovernmentZA.*\n',
    # r'[\s]Attribution..*\n',
    # r'[\s]Elvert Barnes..*\n',
    # r'[\s]Maritza Ríos..*\n', 
    # r'[\s]si.robi.*\n',
    # r'[\s]Steven Depolo.*\n',
    # r'[\s]Brendan Keene.*\n',
    # r'[\s]Francesco Ranieri.*\n',
    # r'[\s]Nathan Keirn.*\n',
    # r'[\s]Giorgio Minguzzi.*\n',
    #r'^[\w+]\.[\w+]\.[\w+]$'
    #r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    
    r' \n',    
]

def merge_paragraphs(text):
    # Split on one or more empty lines
    blocks = re.split(r'\n\s*\n+', text.strip())

    paragraphs = []
    for block in blocks:
        # Remove leading/trailing whitespace from each line
        lines = [line.strip() for line in block.splitlines() if line.strip()]

        # Join lines within the block into a single paragraph
        paragraph = " ".join(lines)

        if paragraph:
            paragraphs.append(paragraph)

    return paragraphs

# Pattern to remove 2-4 words separated by dots.
# The only space can be at the beginnign of the line
pattern = re.compile(
    r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    re.MULTILINE
)

In [2]:
# Specify the URL of the website you want to scrape
url = 'https://easyitaliannews.com/2026/06/'

# To avoid server error: 403
headers = {
    "User-Agent": "Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:109.0) Gecko/20100101 Firefox/117.0"
}

# Send a GET request to the URL
response = requests.get(url, headers=headers)

# Check if the request was successful (status code 200)
if response.status_code == 200:
    # Parse the HTML content using BeautifulSoup
    soup = BeautifulSoup(response.text, 'html.parser')

    l_tds = soup.find_all('td')

    l_of_news = []
    for td in soup.find_all('td'):
        try:
            l_of_news.append(td.find('a').get('href'))
        except:
            pass

In [3]:
%%time
l_of_news

CPU times: total: 0 ns
Wall time: 11.4 μs


['https://easyitaliannews.com/2026/06/02/',
 'https://easyitaliannews.com/2026/06/04/',
 'https://easyitaliannews.com/2026/06/06/',
 'https://easyitaliannews.com/2026/06/09/',
 'https://easyitaliannews.com/2026/06/11/',
 'https://easyitaliannews.com/2026/06/13/',
 'https://easyitaliannews.com/2026/06/16/']

In [6]:
%%time
pattern = re.compile(
    r'^\s*[A-Za-z0-9_-]+(?:\.[A-Za-z0-9_-]+){1,3}\s*$',
    re.MULTILINE
)

for url in l_of_news:
# for url in l_of_news[:1]:
    news_date = url[28:38].replace('/', '-')
    print(news_date)

    # Send a GET request to the URL
    response = requests.get(url, headers=headers)

    # Check if the request was successful (status code 200)
    if response.status_code == 200:
        # Parse the HTML content using BeautifulSoup
        soup = BeautifulSoup(response.text, 'html.parser')

        for h4 in soup.find_all("h4"):
            p = soup.new_tag("p")
            p.string = "<b>" + h4.get_text(strip=True) + "</b>"
        
            h4.replace_with(p)
            
            # Add two line breaks after the paragraph
            p.insert_after(soup.new_tag("br"))
            p.insert_after(soup.new_tag("br"))

        ns = soup.find('div', {'class': 'entry-content'})
            
        # remove <strong> tags
        for strong in ns.find_all("strong"):
            strong.unwrap()

        # Remove <div class="wp-caption alignnone">...</div>
        for div in ns.select("div.wp-caption.alignnone"):
            div.decompose()
            
        # Remove all <a> tags and their contents
        for a in ns.find_all("a"):
            a.decompose()

        # Remove all <img> tags and their contents
        for img in ns.find_all("img"):
            img.decompose()

        # Remove all tags after "Subscribe" - which is a link
        # So, I remove from the next tag!
        for p in ns.find_all("p"):
            if "to receive each bulletin via email (free!)" in p.get_text():
                # remove this paragraph and everything after it
                current = p
                while current:
                    nxt = current.find_next_sibling()
                    current.decompose()
                    current = nxt
                break

        # Extracting all paragraphs
        l_content = []
        #mp3 = ''
        paragraphs = ns.find_all(True)
        for idx, paragraph in enumerate(paragraphs):
            # print(f"Paragraph {idx+1}: {paragraph.text}")
            p = str(paragraph.text)
            l_content.append(p)


        s = ('\n').join(l_content).split('Il tuo aiuto per noi è importante!')[0].split('Subscribe')[0]


        t = ''
        for el in l_to_remove:
            t = re.sub(el, '\n', s)
            s = t

        # # Remove the links to websites (2-4 words separate by dots)
        # cleaned_text = pattern.sub('', s)

        # Group text by paragraphs
        #paragraphed = merge_paragraphs(cleaned_text)
        paragraphed = merge_paragraphs(s)

        # Output with blank line before paragraphs that start with a capitalized word
        result = []
        for i, para in enumerate(paragraphed):
            if re.fullmatch(r'[A-Z]+', para.split()[0]):
                result.append("")  # blank line
            result.append(para)
        
        final_text = "\n\n".join(result)
        
        
        with open(f'EasyItalianNews_{news_date}.txt', 'w', encoding='utf-8') as f:
            f.write(final_text)

        # Find mp3 file
        audio_source = soup.select_one("audio source")
        if audio_source:
            mp3_url = audio_source.get("src").split('?')[0]

        doc = requests.get(mp3_url)
    
        with open(f'EasyItalianNews_{news_date}.mp3', 'wb') as f:
            f.write(doc.content)
        
print('done')

2026-06-02


2026-06-04


In [5]:
result[-5:]

['La vittoria corona una stagione eccezionale, segnata da una clamorosa inversione di tendenza nei playoff.',
 'Infatti, dopo aver perso due delle prime tre partite, i Knicks ne hanno vinte 15 delle successive 16.',
 'Il grande successo è dovuto anche al nuovo allenatore della squadra Mike Brown, capace di rinforzare il gruppo con un gioco rapido e un maggiore utilizzo delle rotazioni.',
 'Nel team vincente c’è anche un pezzo d’Italia grazie all’assistente allenatore Riccardo Fois.',
 'Per New York, da sempre capitale culturale del basket, è finita l’epoca delle delusioni e inizia un periodo di grandi festeggiamenti.']

In [8]:
# Copying text to clipboard

In [22]:
import pyperclip

pyperclip.copy(str(final_text))
print("Copied to clipboard")

Copied to clipboard


In [9]:
# Finding tags

In [26]:
for tag in soup.find_all(True):
    txt = tag.get_text(" ", strip=True)
    if "mp3" in txt or "Emergency" in txt or "dreese" in txt:
        print(tag.name)
        print(tag)
        print("-" * 80)